# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [180]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [181]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [182]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [183]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [135]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [136]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [137]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [138]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [139]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [140]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Perform 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Hold briefly, then lower back down. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\nT

In [141]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—contributes to a stronger immune system, better mood, and improved ability to learn and remember. Poor or insufficient sleep can negatively impact these processes and is associated with various health issues, including weakened immunity, mental health problems, and increased risk of chronic conditions. Therefore, maintaining good sleep hygiene and ensuring restful sleep are essential components of a healthy lifestyle.'

In [142]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using peppermint or lavender essential oils\n- Maintaining a regular sleep schedule\n- Practicing deep breathing exercises\n- Doing progressive muscle relaxation\n- Engaging in grounding techniques by focusing on your senses\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese approaches can help reduce stress and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [143]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [144]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [145]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate and prevent lower back discomfort.'

In [146]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health. Maintaining a regular sleep schedule, creating a comfortable sleep environment, and practicing good sleep hygiene—all help ensure restful, quality sleep. Proper sleep supports immune function, mental health, and nutrient absorption, and helps regulate mood, energy levels, and cognitive performance. Poor sleep or insomnia can disrupt these functions and negatively affect overall well-being.'

In [147]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root can also promote relaxation and help reduce headache symptoms. Additionally, staying well-hydrated, ensuring adequate sleep, managing stress levels, and avoiding known triggers like certain foods or eye strain can be effective in managing headaches and stress naturally.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:
BM25 is based on word similarity over semantic meaning which embeddings specialize. 

BM25 will outperform embedding this these scenarios where keyword match pays a crucual role. For example product names, acronyms, serial numbers, error codes, or highly specialized medical/technical jargon. It is best suited for queries such as "What are the side effects of Lisinopril 20mg ?"


Emebedding model is semantic meaning based which means it will return documents related to Blood Pressure as they live in similar scemantic space
where as BM25 will give high score to documnet containing Lisinopril 20mg and fetch the docuemnt. 

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [148]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [149]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [150]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, then repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate and prevent lower back pain.'

In [151]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It supports physical recovery by allowing the body to repair tissues and regenerate cells during deep sleep stages. Sleep also aids in consolidating memories and learning, which benefits mental well-being. Additionally, during sleep, the body releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—is essential for maintaining mood, cognitive function, and overall physical health. Poor sleep or sleep disorders like insomnia can negatively impact these processes and overall well-being.'

In [152]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, doing progressive muscle relaxation, using grounding techniques, taking short walks in nature, listening to calming music, staying hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [153]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [154]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [155]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises are gentle and focused on stretching and strengthening the lower back area, w

In [156]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep supports immune function, helps prevent chronic health issues, improves mood, and enhances learning and memory. Therefore, maintaining good sleep hygiene and ensuring sufficient sleep duration (7-9 hours per night) are essential for overall health and wellness.'

In [157]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Consuming caffeine in small amounts (with caution)\n- Practicing deep breathing exercises, like box breathing\n- Doing progressive muscle relaxation to release tension\n- Engaging in grounding techniques, such as naming objects around you\n- Taking short walks in nature\n- Listening to calming music\n\nAdditionally, maintaining a regular sleep schedule, managing stress through mindfulness or meditation, and practicing relaxation exercises can help manage stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall by overcoming the "vocabulary mismatch" problem.

When a user asks a question, they typically use one specific set of words. However, the relevant information in the database might be written using completely different terminology, synonyms, or phrasing. If we only search using the original prompt, the retriever might miss highly relevant documents simply because the semantic overlap isn't strong enough.

Because you are casting a wider net and pooling the results, the system is much more likely to retrieve the relevant documents that would have otherwise slipped through the cracks, thereby increasing the overall recall (the percentage of total relevant documents successfully retrieved).


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [158]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [159]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [160]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [161]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [162]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [163]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate arching your back upward (cat) and sagging it downward (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest, hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [164]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—ensures that these processes occur effectively. Poor or insufficient sleep can impact immune function, increase stress levels, impair memory and concentration, and contribute to various health issues. Therefore, maintaining good sleep habits and hygiene is essential for promoting optimal health and well-being.'

In [165]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, taking short walks in nature, listening to calming music, and practicing mindfulness or meditation. For headaches specifically, remedies such as staying hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, and using essential oils like peppermint or lavender can be helpful.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [166]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [167]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [168]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, arms crossed over your chest. Tighten your stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and focused on stretching and st

In [169]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting adequate sleep—typically 7 to 9 hours per night—is associated with a stronger immune system, better mood, improved concentration, and overall vitality. Poor sleep or sleep difficulties, such as insomnia, can lead to health problems including fatigue, stress, weakened immunity, and increased risk for chronic conditions. Maintaining good sleep hygiene and creating a restful sleep environment are important for ensuring quality sleep and supporting overall health.'

In [170]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For headaches:\n  - Drink water to stay hydrated\n  - Apply cold or warm compresses to the head or neck\n  - Rest in a dark, quiet room\n  - Gently massage the temples and neck\n  - Use peppermint or lavender essential oils\n  - Maintain a regular sleep schedule\n  - Consume caffeine in small amounts (with caution)\n\n- For stress relief:\n  - Practice deep breathing exercises (inhale for 4 counts, hold, exhale)\n  - Try progressive muscle relaxation by tensing and releasing muscle groups\n  - Use grounding techniques, such as naming things you see, hear, feel, smell, and taste\n  - Take short walks, preferably in nature\n  - Listen to calming music\n\nThese methods can help manage stress and headaches naturally and effectively.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [171]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [172]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [173]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [174]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [175]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [176]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back upward (cat) and letting it sag downward (cow). Perform 10-15 repetitions.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds, and repeat 8-12 times.\n\nStarting with gentle stretching and strengthening exercises like these can help alleviate lower back discomfort and prevent future issues. As always, it’s best to consult with a healthcare professional befor

In [177]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. According to the provided information, during sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and sleep occurs in cycles that include REM and non-REM stages, each serving different functions such as deep body repair and brain activity for memory and learning. Good sleep quality can be promoted through proper sleep hygiene practices like maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment. Adequate sleep is essential for maintaining a healthy immune system, managing stress, and supporting metabolic health, thereby contributing significantly to overall wellbeing.'

In [178]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For stress:\n  - Deep breathing exercises (e.g., inhaling for 4 counts, holding, exhaling, and holding)\n  - Progressive muscle relaxation (tensing and releasing muscle groups)\n  - Grounding techniques (e.g., naming things you see, hear, feel, smell, taste)\n  - Taking short walks, especially in nature\n  - Listening to calming music\n  - Practicing mindfulness and meditation (focused attention, body scan, loving-kindness)\n  - Engaging in hobbies and maintaining social connections\n\n- For headaches:\n  - Staying hydrated by drinking water\n  - Applying cold or warm compresses to the head or neck\n  - Resting in a dark, quiet room\n  - Gently massaging the temples and neck\n  - Using essential oils such as peppermint or lavender\n  - Maintaining a regular sleep schedule\n  - Managing stress effectively\n\nThese approaches emphasize relaxation, hydration, and mindfulness, which can help alleviate stress and headache symptom

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If we apply this to an FAQ where sentences are short and highly repetitive, the embeddings will be very similar across the board. Because the semantic distances between sentences are so small, the algorithm might fail to find clear breakpoints. This usually results in either 
1. lumping the entire FAQ section into one massive, 
2. or making arbitrary, fragmented splits based on microscopic noise in the embeddings.

Adjust the alogrithm: 
1. We could apply gradiant or standard_deviation might help the algorithm detect relative shifts in meaning (e.g., the transition from one specific question to the next) rather than relying on the overall distance distribution.
2. May be sematic chuking is no the right streategy for FAQs, Multi-Query Retriever or Parent-Document Retrieval might be better fit
- Multi-Query Retrievers: By generating multiple reformulations, you bridge the gap between the user's unpredictable phrasing and the official, static phrasing of the FAQ document. It retrieves documents for each unique query and combines them, greatly increasing the chances of finding the right FAQ entry.
- Parent-Document Retrieval: By generating multiple reformulations, you bridge the gap between the user's unpredictable phrasing and the official, static phrasing of the FAQ document. It retrieves documents for each unique query and combines them, greatly increasing the chances of finding the right FAQ entry.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

In [179]:
!python -m pip install ragas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [206]:


if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your Langchain API Key:")


In [210]:
import os
from langsmith import Client
import sys

# Ragas imports
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.run_config import RunConfig
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


#--------------------------------
# Workaround for VS Code/Cursor widget rendering issue
# Disable widget-based progress bars
os.environ["JUPYTER_WIDGETS_ENABLED"] = "0"

# Configure tqdm (used by Ragas) to use console output instead of notebook widgets
try:
    from tqdm.auto import tqdm
    # Force tqdm to detect we're in a non-widget environment
    # This makes tqdm use text-based progress bars
    import tqdm as tqdm_module
    # Override notebook detection
    tqdm_module.tqdm.notebook = False
except (ImportError, AttributeError):
    pass

# Suppress widget rendering errors by catching display exceptions
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='ipywidgets')

print("✓ Widget rendering workaround applied - progress bars will use text output")
#--------------------------------

# --- SETUP: Enable LangSmith Tracing ---
os.environ["LANGCHAIN_TRACING_V2"] = "true"
client = Client()


#----------Generating Golden Dataset----------------------

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Load your documents
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

print("Starting Step 1: Generating Golden Dataset...")

# Initialize the generator with the properly wrapped LLM
generator = TestsetGenerator(llm=eval_llm, embedding_model=eval_embeddings)
testdataset = generator.generate_with_langchain_docs(docs, testset_size=10)




#-------------- Evaluating Retrievers ------------------
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
import time
from langchain_core.tracers.context import tracing_v2_enabled

def evaluate_retriever_with_metrics(retriever, retriever_name, raw_testdataset):
    project_name = f"Eval_{retriever_name.replace(' ', '_')}"
    
    # 1. Convert Ragas Testset directly to a list of dictionaries
    data_list = raw_testdataset.to_list()
    
    # Extract lists dynamically based on the dictionary keys
    queries = [item.get("user_input", item.get("question")) for item in data_list]
    references = [item.get("reference", item.get("ground_truth")) for item in data_list]
    
    contexts_list = []
    
    # 2. Retrieve documents using the Context Manager to FORCE the project name
    with tracing_v2_enabled(project_name=project_name):
        for query in queries:
            retrieved_docs = retriever.invoke(query)
            contexts_list.append([doc.page_content for doc in retrieved_docs])

    # 3. Format data for Ragas evaluate()
    data = {
        "user_input": queries,           
        "retrieved_contexts": contexts_list, 
        "reference": references,         
        "question": queries,             
        "contexts": contexts_list,       
        "ground_truth": references,      
        "answer": [""] * len(queries)
    }
    
    # 4. Run the evaluation
    # Configure Ragas to slow down to prevent TimeoutErrors
    run_config = RunConfig(max_workers=2, max_retries=5)
    ragas_result = evaluate(
        dataset=Dataset.from_dict(data),
        metrics=[context_precision, context_recall],
        llm=eval_llm,               
        embeddings=eval_embeddings, 
        run_config=run_config
    )
    
    # 5. Fetch Latency and Cost (with a buffer and safety net!)
    print(f"Waiting 3 seconds for LangSmith to process {project_name} logs...")
    time.sleep(3)
    
    total_latency = 0
    total_cost = 0.0
    runs = []
    
    try:
        runs = list(client.list_runs(project_name=project_name, execution_order=1))
        for run in runs:
            if run.end_time and run.start_time:
                total_latency += (run.end_time - run.start_time).total_seconds()
            if hasattr(run, 'total_cost') and run.total_cost is not None:
                total_cost += float(run.total_cost)
    except Exception as e:
        print(f"Warning: Could not fetch LangSmith metrics for {project_name}. Error: {e}")
            
    avg_latency = total_latency / len(runs) if runs else 0
    
    return {
        "Retriever": retriever_name,
        "Context Precision": ragas_result["context_precision"],
        "Context Recall": ragas_result["context_recall"],
        "Avg Latency (s)": round(avg_latency, 4),
        "Total Cost ($)": round(total_cost, 6)
    }


#------------- Evaluating Retrievers -------------------
my_retrievers = {
    "Naive Vector": naive_retriever,
    "BM25": bm25_retriever,
    "Multi-Query": multi_query_retriever, 
    "Ensemble": ensemble_retriever       
}

compiled_results = []

for name, retriever in my_retrievers.items():
    print(f"Running evaluation for {name}...")
    metrics = evaluate_retriever_with_metrics(retriever, name, testdataset)
    compiled_results.append(metrics)

# Print the final compiled list
print("\n--- FINAL COMPILED LIST ---")
for result in compiled_results:
    print(f"\nRetriever: {result['Retriever']}")
    print(f"  - Context Precision: {result['Context Precision']}")
    print(f"  - Context Recall: {result['Context Recall']}")
    print(f"  - Avg Latency: {result['Avg Latency (s)']}s")
    print(f"  - Total Cost: ${result['Total Cost ($)']}")


✓ Widget rendering workaround applied - progress bars will use text output
Starting Step 1: Generating Golden Dataset...


/var/folders/n6/6n8v0bz157n43frt7yxyspqm0000gn/T/ipykernel_77653/3546009097.py:10: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/var/folders/n6/6n8v0bz157n43frt7yxyspqm0000gn/T/ipykernel_77653/3546009097.py:10: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall
/var/folders/n6/6n8v0bz157n43frt7yxyspqm0000gn/T/ipykernel_77653/3546009097.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; 

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Waiting 3 seconds for LangSmith to process Eval_Multi-Query logs...
Running evaluation for Ensemble...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception raised in Job[3]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-jO4MdaxW728Tt29IybLtmXt1 on tokens per min (TPM): Limit 30000, Used 28809, Requested 3355. Please try again in 4.328s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})


Waiting 3 seconds for LangSmith to process Eval_Ensemble logs...

--- FINAL COMPILED LIST ---

Retriever: Naive Vector
  - Context Precision: [0.9999999999, 0.3333333333, 0.49999999995, 0.9999999999]
  - Context Recall: [0.3333333333333333, 1.0, 1.0, 1.0]
  - Avg Latency: 0.3223s
  - Total Cost: $0.0

Retriever: BM25
  - Context Precision: [0.9999999999, 0.9999999999, 0.0, 0.0]
  - Context Recall: [0.3333333333333333, 0.5, 0.0, 0.0]
  - Avg Latency: 0.0008s
  - Total Cost: $0.0

Retriever: Multi-Query
  - Context Precision: [0.9999999999, 0.249999999975, 0.9999999999, 0.9999999999]
  - Context Recall: [0.3333333333333333, 1.0, 1.0, 1.0]
  - Avg Latency: 1.7691s
  - Total Cost: $0.000405

Retriever: Ensemble
  - Context Precision: [0.5909090908795455, 0.5909090908795455, 0.5714285713999999, 0.5833333333041667]
  - Context Recall: [1.0, nan, 1.0, 1.0]
  - Avg Latency: 3.844s
  - Total Cost: $0.000408


Final Analysis:

Naive Vector is the best overall choice. It offers the most optimal balance between performance, speed, and cost. It matches the excellent Context Recall of the much slower and more expensive Multi-Query method (hitting 1.0 on 3 out of 4 queries) while maintaining perfectly respectable precision. More importantly, it achieves this with a very low average latency of ~0.32 seconds and entirely for free ($0.0). While BM25 is technically faster, its accuracy completely collapsed on half the queries. Conversely, Multi-Query offers slightly better precision but introduces LLM API costs and is over 5 times slower, making Naive Vector the most efficient and robust choice for production